In [45]:
import importlib
import spark_data_check
importlib.reload(spark_data_check)

<module 'spark_data_check' from '/home/jupyter-ykim68@ncsu.edu/Project2/spark_data_check.py'>

In [46]:
import os
print(os.listdir('/home/jupyter-ykim68@ncsu.edu/Project2'))

['__pycache__', 'air.csv', 'Project2.ipynb', 'spark_data_check.py', '.ipynb_checkpoints']


In [47]:
from spark_data_check import SparkDataCheck
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()   #build SparkSession
obj = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")  #load air.csv file

## Preliminary Check: Test Class #1
Instruction: Create a method that checks if each value in a numeric column is within user defined limits (upper and lower bounds, inclusive) and returns the dataframe with an appended column of Boolean values.

In [48]:
obj.df.show(5)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-24 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-24 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-24 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-24 21:00:00|   2.2|       137

26/03/24 17:11:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ykim68@ncsu.edu/Project2/air.csv


In [49]:
print(obj.df.dtypes)

[('_c0', 'int'), ('Date', 'string'), ('Time', 'timestamp'), ('CO(GT)', 'double'), ('PT08.S1(CO)', 'int'), ('NMHC(GT)', 'int'), ('C6H6(GT)', 'double'), ('PT08.S2(NMHC)', 'int'), ('NOx(GT)', 'int'), ('PT08.S3(NOx)', 'int'), ('NO2(GT)', 'int'), ('PT08.S4(NO2)', 'int'), ('PT08.S5(O3)', 'int'), ('T', 'double'), ('RH', 'double'), ('AH', 'double')]


Before applying the method, I examined the data types of each column to identify which columns are numeric. The output shows that several columns, including `CO(GT)`, `T`, `RH`, and `AH`, are recognized as numeric types (double). Therefore, these columns are appropriate candidates for applying the numeric range validation method.

### Test: Numeric range method

I test the `check_numeric_range()` method to verify that it correctly evaluates whether values in a numeric column fall within user-defined bounds. 
This test ensures that the method properly handles valid numeric columns, applies inclusive bounds, and appends a Boolean column indicating whether each value satisfies the specified condition.

In [50]:
obj.check_numeric_range("T", lower=0, upper=40)
obj.df.select("T", "T_in_range").show(10)

+----+----------+
|   T|T_in_range|
+----+----------+
|13.6|      true|
|13.3|      true|
|11.9|      true|
|11.0|      true|
|11.2|      true|
|11.2|      true|
|11.3|      true|
|10.7|      true|
|10.7|      true|
|10.3|      true|
+----+----------+
only showing top 10 rows


After applying the `check_numeric_range()` method to the `T` column with lower and upper bounds of 0 and 40, a new Boolean column (`T_in_range`) was successfully sppended to the DataFrame. The results indicate that all displayed values fall within the specified range, and therefore the corresponding Boolean values are `True`.
The results confirm that the method correctly identifies whether values fall within the specified bounds, appends a Boolean column, and preserves the original DataFrame structure.

### Test: String level method
I test the `check_string_levels()` method to verify that it correctly evaluates whetehr values in a string column belong to a user-defined set of levels. This test ensures that the method properly identifies valid string values, handles non-matching values appropriately, and appends a Boolean column indicating whether each value satisfies the specified condition.

In [51]:
obj.check_string_levels("Date", ["10/03/2004"])
obj.df.select("Date", "Date_valid").show(10)

+---------+----------+
|     Date|Date_valid|
+---------+----------+
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
+---------+----------+
only showing top 10 rows


After applying the `check_string_levels()` method to the `Date` column with the specified level `"10/03/2004"`, a new Boolean column (`Date_valid`) was successfully appended to the DataFrame. The result show that all displayed values are amrked as `False`, indicating that none of the values in the `Data` column match the specified level.
This outcome suggests that the actual data format in the dataset differs from the provided level.

In [52]:
obj.check_string_levels("Date", ["3/10/2004"])
obj.df.select("Date", "Date_valid").show(10)

+---------+----------+
|     Date|Date_valid|
+---------+----------+
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
+---------+----------+
only showing top 10 rows


The dataset contains values such as `"3/10/2004"`, which do not match `"10/03/2004"` due to differences in formatting. The method correctly identifies that these values are not included in the specified set.

### Test: Missing Value Check Method
I test the `check_missing()` method to verify that it correctly identifies whether each value in a selected column is missing (`NULL`). This test ensures that the method properly appends a Boolean column indicating missingness without altering the original values in the DataFrame.

In [53]:
obj.check_missing("CO(GT)")
obj.df.select("CO(GT)", "CO(GT)_is_null").show(10)

+------+--------------+
|CO(GT)|CO(GT)_is_null|
+------+--------------+
|   2.6|         false|
|   2.0|         false|
|   2.2|         false|
|   2.2|         false|
|   1.6|         false|
|   1.2|         false|
|   1.2|         false|
|   1.0|         false|
|   0.9|         false|
|   0.6|         false|
+------+--------------+
only showing top 10 rows


After applying the `check_missing()` method to the `CO(GT)` column, a new Boolean column (`CO(GT)_is_null`) was successfully appended to the DataFrame. The displayed results show `False` for all visible rows, indicating that none of the values in this subset are missing.
This confirms that the method correctly detects missing values, appends the expected Boolean output column, and preserves the original DataFrame structure.